# Automated LiDAR → DEM / DSM Processing Workflow

**Author:** Pamela A. Green 

**Date:** June 27, 2026

---

## Overview

This notebook implements a **reproducible, tile‑based LiDAR processing workflow** that converts raw airborne LiDAR point clouds (LAS/LAZ) into aligned raster terrain and surface products, including a **Digital Elevation Model (DEM)**, **Digital Surface Model (DSM)**, and a **normalized DSM (nDSM)** suitable for above‑ground structure analysis. It automates preprocessing, ground classification, rasterization, gap filling, and basic QA so that large LiDAR datasets can be processed consistently and inspected easily before use in GIS or modeling workflows.

The objective is to **automate LiDAR‑to‑raster processing** in a way that is consistent, repeatable, and scalable across datasets, while remaining transparent and easy to validate at each processing stage.

---

## Purpose and Design Goals

This workflow is designed to:
* Standardize LiDAR **tiling, preprocessing, and ground classification** for large point clouds. 
* Reliably separate **ground** and **non-ground** returns using Cloth Simulation Filtering (CSF) with configurable presets. 
* Generate DEM and DSM products using **LiDAR-native TIN interpolation**, with explicit control over returns and class filters. 
* Produce both raw and **void-filled** DEM mosaics, plus a **fill mask**, to make gap handling transparent and reproducible. 
* Ensure DEM and DSM mosaics are **grid-aligned** before nDSM computation, avoiding subtle pixel alignment errors. 
* Enforce explicit and consistent **CRS handling** by reading the LAS/LAZ CRS when available and falling back to a user-defined CRS when needed. 
* Capture lightweight **tile-level QA summaries** (e.g., ground/non-ground ratios) to support rapid screening of classification performance. 

All processing steps are implemented using **open-source, pip-installable Python libraries**, emphasizing numerical robustness and compatibility with common GIS, hydrologic, and spatial analysis tools.


---

## Workflow Summary

At a high level, the notebook proceeds through the following stages:

1. **Tiling and buffering** of large LiDAR files into manageable tiles with metadata and footprints, suitable for chunked processing. 
2. **Data ingestion and CRS handling**, including reading CRS from the LAS/LAZ header or applying a fallback CRS if none is present. 
3. **Preprocessing and outlier filtering** using a local, spatially aware elevation filter (with optional global guards) to remove obvious noise while preserving terrain and structures. 
4. **Ground / non-ground classification** via Cloth Simulation Filtering (CSF) with named presets (e.g., default, urban, forest, steep), writing a classified LAS per tile and summarizing class statistics. 
5. **Rasterization of DEM and DSM** per tile using TIN-based gridding, with separate controls for DEM returns/classes and DSM returns/classes. 
6. **Mosaicking of tile rasters** into seamless DEM and DSM products, followed by a grid alignment step that aligns DSM to the DEM grid when necessary. 
7. **DEM gap handling and nDSM derivation**, including:
    * writing a **raw DEM mosaic**,
    * generating a **DEM void mask**,
    * creating a **filled DEM** via nearest-neighbor gap filling, and
    * computing a **final nDSM** from either the raw or filled DEM, based on a user-set option. 
8. **Cleanup and optional QA/visualization**, including removal of temporary tiles and intermediate rasters and optional exploratory plots for smaller datasets. 

The notebook is intended to be **modular and extensible**, so users can adjust tiling parameters, CSF presets, interpolation settings, and QA logic without changing the core tiling → classification → rasterization → mosaicking structure. Users may also swap in alternative interpolation methods, add PDAL‑based steps, or extend QA and visualization without changing the core logic. The example dataset (BL6_UTM_32_ort_337.laz) is from the *LiDAR and Photogrammetry Point Clouds and R code for University of Padova courses on Geomatics for Forest and Environmental Sciences* available at https://doi.org/10.5281/zenodo.17934978, downloaded on June 24, 2026.

## Environment setup

Loads the core Python libraries used throughout the workflow for LiDAR I/O, tiling, filtering, classification, raster generation, mosaicking, and QA. Dependencies can be installed from a requirements.txt file inside a virtual environment for a reproducible setup.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
import time
import math
from pathlib import Path

import contextlib

from pyproj import CRS

import laspy
import geopandas as gpd
import whitebox_workflows as wbw
import CSF_3DFin as CSF
import gc

from IPython.display import display

import xarray as xr
from rasterio.enums import Resampling
from rioxarray.merge import merge_arrays
import rioxarray as rxr

from tqdm import tqdm   # optional, for progress bar
from scipy.spatial import cKDTree
from scipy.ndimage import distance_transform_edt
from shapely.geometry import box

## Runtime tracking

Starts a simple timer to track total workflow runtime from beginning to end. This is useful for benchmarking and testing, but it can be commented out if timing information is not needed.

In [2]:
start = time.time()

## Configuration and paths

Defines the project paths, clears prior outputs, and sets the main processing parameters used throughout the workflow, including tiling, preprocessing, CSF classification, raster generation, CRS handling, and DEM void-filling options. Intermediate files are written to working subdirectories, while final raster products are written to the main raster output folder. 

The example LiDAR dataset used in this notebook ('BL6_UTM_32_ort_337.laz' point cloud dataset) is from the LiDAR and Photogrammetry Point Clouds and R code for University of Padova courses on Geomatics for Forest and Environmental Sciences available at https://doi.org/10.5281/zenodo.17934978, downloaded on June 24, 2026.

In [3]:
PROJECT_ROOT = os.getcwd()

# ----------------------------------------------------------
# Define input LiDAR file
# ----------------------------------------------------------
# ⚠️  IMPORTANT USER ACTION REQUIRED  ⚠️
# Set LIDAR_FILENAME to the name of YOUR .las or .laz file, including the
# extension. The file is expected to be in the data/raw/ folder at the root
# of this repository, e.g. <repo>/data/raw/your_file.laz.
#
# LIDAR_INPUT_PATH is built from PROJECT_ROOT (the notebook's working
# directory). If you keep your data elsewhere, edit LIDAR_INPUT_PATH directly.
#
# NOTE: "BL6_UTM_32_ort_337.laz" is the sample dataset used in this
# repository. Replace it with your own file name.

LIDAR_FILENAME ='BL6_UTM_32_ort_337.laz'
LIDAR_INPUT_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', LIDAR_FILENAME)
local_path = LIDAR_INPUT_PATH

# Directory containing the LiDAR tiles
lidar_parent_dir = os.path.dirname(LIDAR_INPUT_PATH)
lidar_tile_dir = 'lidar_tiles'
lidar_basename = os.path.splitext(os.path.basename(LIDAR_INPUT_PATH))[0]
TILE_DIR = os.path.join(lidar_parent_dir, lidar_tile_dir)
TILE_METADATA_CSV = os.path.join(TILE_DIR, "tile_index.csv")
TILE_FOOTPRINT_GPKG = os.path.join(TILE_DIR, "tile_footprints.gpkg")
TILE_FOOTPRINT_LAYER = "tile_footprints"
os.makedirs(TILE_DIR, exist_ok=True)

# Define raster output directory
RASTER_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed', 'rasters')
os.makedirs(RASTER_DIR, exist_ok=True)

# ----------------------------------------------------------
# Define Coordinate Reference System (explicit and enforced)
# ----------------------------------------------------------
# ⚠️  IMPORTANT USER ACTION REQUIRED  ⚠️
# The workflow reads the CRS from the LAS/LAZ header. If the header has none,
# FALLBACK_CRS is used instead, so you must set it to match your own data.
#
#   1. Check your data's metadata or delivery documentation for the CRS.
#   2. Set FALLBACK_CRS to the matching EPSG code, e.g. "EPSG:32618".
#   3. Set FALLBACK_CRS = None to stop the workflow if no CRS is found
#
# NOTE: "EPSG:7791" is specific to the sample dataset used in this repository.
# Do not reuse it unless your data uses the same CRS. A wrong CRS will
# silently produce misplaced rasters.

FALLBACK_CRS = "EPSG:7791"

# ----------------------------------------------------------
# Safely clear directory contents without deleting the folder (OneDrive-safe)
# ----------------------------------------------------------

def clear_directory_contents(path):
    """
    Remove all files and subdirectories inside `path`
    without deleting `path` itself (OneDrive-safe).
    """
    for name in os.listdir(path):
        full_path = os.path.join(path, name)
        try:
            if os.path.isdir(full_path):
                shutil.rmtree(full_path)
            else:
                os.remove(full_path)
        except PermissionError as e:
            print(f"⚠️ Could not delete {full_path}: {e}")

# ----------------------------------------------------------
# Directories to clear (OneDrive-safe)
# ----------------------------------------------------------

DIRS_TO_CLEAR = [
    RASTER_DIR,
    TILE_DIR,
]

for d in DIRS_TO_CLEAR:
    if os.path.exists(d):
        print(f"Clearing contents of: {d}")
        clear_directory_contents(d)
    else:
        print(f"Directory does not exist: {d}")

print("✅ Directory contents cleared.")

# ----------------------------------------------------------
# Tile parameters (needed for large LiDAR)
# ----------------------------------------------------------

TILE_WIDTH = 250     # meters
TILE_HEIGHT = 250    # meters
TILE_BUFFER = 25     # meters (edge safety for interpolation)

# ----------------------------------------------------------
# Preprocessing / outlier filter parameters
# ----------------------------------------------------------

PREPROCESS_USE_LOCAL_FILTER = True

# k-nearest-neighbor size for local z comparison
PREPROCESS_K = 12

# remove points whose z differs too much from the local neighborhood median
PREPROCESS_LOCAL_Z_THRESH = 8.0   # meters

# robust MAD-based threshold multiplier
PREPROCESS_MAD_MULTIPLIER = 6.0

# optional broad global guardrail for obviously bad points
PREPROCESS_GLOBAL_ZMIN = None
PREPROCESS_GLOBAL_ZMAX = None

# ----------------------------------------------------------
# CSF (Cloth Simulation Filtering) parameters
# ----------------------------------------------------------

CSF_PRESETS = {
    "default": {
        "cloth_resolution": 1.0,
        "class_threshold": 0.05
    },
    "urban": {
        "cloth_resolution": 0.75,
        "class_threshold": 0.04
    },
    "forest": {
        "cloth_resolution": 1.5,
        "class_threshold": 0.08
    },
    "steep": {
        "cloth_resolution": 0.75,
        "class_threshold": 0.06
    }
}

CSF_ACTIVE_PRESET = "default"

CSF_PARAMS = CSF_PRESETS[CSF_ACTIVE_PRESET]
CSF_CLOTH_RESOLUTION = CSF_PARAMS["cloth_resolution"]
CSF_CLASS_THRESHOLD = CSF_PARAMS["class_threshold"]

# ----------------------------------------------------------
# DEM / DSM rasterization parameters
# ----------------------------------------------------------

DEM_RETURNS_INCLUDED = "all"
DEM_EXCLUDED_CLASSES = "0,1,5,6,7,18"

DSM_RETURNS_INCLUDED = "all"
DSM_EXCLUDED_CLASSES = "7,18"

RASTER_WRITE_COMPRESS = "LZW"
RASTER_WRITE_DTYPE = "float32"

RASTER_RESOLUTION = 1.0    # meters

# ----------------------------------------------------------
# DEM void filling / output selection parameters
# ----------------------------------------------------------

WRITE_RAW_DEM = True
WRITE_FILLED_DEM = True
WRITE_FILL_MASK = True

NDSM_SOURCE_DEM = "filled"   # options: "raw" or "filled"

# ----------------------------------------------------------
# File naming conventions
# ----------------------------------------------------------

DEM_SUFFIX = "_dem"
DSM_SUFFIX = "_dsm"
NDSM_SUFFIX = "_ndsm"

print("Configuration loaded.")
print(f"LiDAR input: {LIDAR_INPUT_PATH}")
print(f"LiDAR Tiles: {TILE_DIR}")
print(f"Tile size: {TILE_WIDTH} x {TILE_HEIGHT} m")
print(f"Raster resolution: {RASTER_RESOLUTION} m")

Clearing contents of: /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/processed/rasters
Clearing contents of: /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/raw/lidar_tiles
✅ Directory contents cleared.
Configuration loaded.
LiDAR input: /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/raw/BL6_UTM_32_ort_337.laz
LiDAR Tiles: /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/raw/lidar_tiles
Tile size: 250 x 250 m
Raster resolution: 1.0 m


## LiDAR metadata and tiling check
Reads LiDAR header metadata to inspect dataset size, bounds, point format, and CRS before full processing begins. This step also determines whether tiling is required based on point count and provides a quick check that the dataset is safe to process as configured. For smaller LiDAR datasets, tiling may be unnecessary, but for large point clouds it is a useful safeguard for memory management and stable processing.

In [4]:
# ----------------------------------------------------------
# LiDAR metadata inspection (NO full data load)
# ----------------------------------------------------------

print("Inspecting LiDAR metadata only (no point arrays loaded)...")

with laspy.open(LIDAR_INPUT_PATH) as fh:
    header = fh.header
    point_count = header.point_count
    point_format = header.point_format
    bounds = (
        header.min[0], header.max[0],
        header.min[1], header.max[1],
        header.min[2], header.max[2],
    )

    parsed_crs = header.parse_crs()

if parsed_crs is not None:
    LIDAR_CRS = CRS.from_user_input(parsed_crs)
elif FALLBACK_CRS is not None:
    LIDAR_CRS = CRS.from_user_input(FALLBACK_CRS)
else:
    raise ValueError("No CRS found in LAS/LAZ header and no FALLBACK_CRS provided.")

LIDAR_CRS_STR = LIDAR_CRS.to_string()

print(f"Point count: {point_count:,}")
print(f"Point format: {point_format}")
print("Bounds (xmin, xmax, ymin, ymax, zmin, zmax):")
print(bounds)
print(f"Detected CRS: {LIDAR_CRS_STR}")

# ----------------------------------------------------------
# Safety check: enforce tiling for large datasets
# ----------------------------------------------------------

MAX_POINTS_NO_TILING = 10_000_000  # conservative threshold

if point_count > MAX_POINTS_NO_TILING:
    print(
        "\n⚠️  Large LiDAR dataset detected.\n"
        f"Point count ({point_count:,}) exceeds safe single‑pass limit.\n"
        "Tiling is REQUIRED before preprocessing, CSF, or rasterization.\n"
        "Proceed to the tiling cell next."
    )
else:
    print(
        "\n✅ Dataset size is within single‑pass limits.\n"
        "You may proceed without tiling (not recommended for production)."
    )

Inspecting LiDAR metadata only (no point arrays loaded)...
Point count: 4,256,603
Point format: <PointFormat(1, 0 bytes of extra dims)>
Bounds (xmin, xmax, ymin, ymax, zmin, zmax):
(np.float64(759340.7433574634), np.float64(759840.7423574633), np.float64(5148495.495858612), np.float64(5148995.494858611), np.float64(761.881), np.float64(931.327))
Detected CRS: EPSG:7791

✅ Dataset size is within single‑pass limits.
You may proceed without tiling (not recommended for production).


## LiDAR tiling and buffering

Splits large LiDAR datasets into manageable processing tiles and applies a buffer around each tile to reduce edge effects during classification and raster interpolation. The tiling step also records tile metadata and footprints, which helps with tracking coverage and debugging larger workflows. Because tiling only needs to be done once for a given input and tile scheme, this section can be skipped or run separately when testing later processing steps.

In [5]:
def tile_las_by_bounds_buffered(
    input_path,
    output_dir,
    tile_width,
    tile_height=None,
    tile_buffer=0,
    chunk_size=1_000_000,
    min_points_per_tile=1,
    metadata_csv=None,
    footprint_gpkg=None,
    footprint_layer="tile_footprints",
    crs=None,
):
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if tile_height is None:
        tile_height = tile_width

    tile_records = []

    with laspy.open(input_path) as src:
        header = src.header
        x_min, y_min, _ = header.mins
        x_max, y_max, _ = header.maxs

        n_tiles_x = math.ceil((x_max - x_min) / tile_width)
        n_tiles_y = math.ceil((y_max - y_min) / tile_height)

        print(f"x range: {x_min} to {x_max}")
        print(f"y range: {y_min} to {y_max}")
        print(f"tile grid: {n_tiles_x} x {n_tiles_y}")
        print(f"tile buffer: {tile_buffer}")

        tile_defs = []
        for ix in range(n_tiles_x):
            for iy in range(n_tiles_y):
                core_xmin = x_min + ix * tile_width
                core_xmax = min(core_xmin + tile_width, x_max + 1e-9)
                core_ymin = y_min + iy * tile_height
                core_ymax = min(core_ymin + tile_height, y_max + 1e-9)

                buf_xmin = max(x_min, core_xmin - tile_buffer)
                buf_xmax = min(x_max + 1e-9, core_xmax + tile_buffer)
                buf_ymin = max(y_min, core_ymin - tile_buffer)
                buf_ymax = min(y_max + 1e-9, core_ymax + tile_buffer)

                tile_id = f"tile_{ix}_{iy}"
                tile_path = output_dir / f"{tile_id}_buf.las"

                tile_defs.append({
                    "tile_id": tile_id,
                    "ix": ix,
                    "iy": iy,
                    "core_xmin": core_xmin,
                    "core_xmax": core_xmax,
                    "core_ymin": core_ymin,
                    "core_ymax": core_ymax,
                    "buf_xmin": buf_xmin,
                    "buf_xmax": buf_xmax,
                    "buf_ymin": buf_ymin,
                    "buf_ymax": buf_ymax,
                    "tile_path": str(tile_path),
                })

        writers = {}
        counts = {}

        try:
            total_points_seen = 0

            for chunk_num, points in enumerate(src.chunk_iterator(chunk_size), start=1):
                x = points.x.copy()
                y = points.y.copy()

                total_points_seen += len(points)
                print(f"Processing chunk {chunk_num} with {len(points)} points...")

                for td in tile_defs:
                    mask = (
                        (x >= td["buf_xmin"]) & (x < td["buf_xmax"]) &
                        (y >= td["buf_ymin"]) & (y < td["buf_ymax"])
                    )

                    if not np.any(mask):
                        continue

                    sub_points = points[mask]
                    tile_key = td["tile_id"]

                    if tile_key not in writers:
                        writers[tile_key] = laspy.open(td["tile_path"], mode="w", header=header)
                        counts[tile_key] = 0

                    writers[tile_key].write_points(sub_points)
                    counts[tile_key] += len(sub_points)

        finally:
            for writer in writers.values():
                writer.close()

        kept_records = []
        for td in tile_defs:
            count = counts.get(td["tile_id"], 0)
            if count < min_points_per_tile:
                if os.path.exists(td["tile_path"]):
                    os.remove(td["tile_path"])
                continue

            td["point_count"] = count
            td["tile_width"] = tile_width
            td["tile_height"] = tile_height
            td["tile_buffer"] = tile_buffer
            kept_records.append(td)

    tile_df = pd.DataFrame(kept_records)

    if metadata_csv:
        tile_df.to_csv(metadata_csv, index=False)

    if footprint_gpkg and not tile_df.empty:
        gdf = gpd.GeoDataFrame(
            tile_df.copy(),
            geometry=[
                box(r.core_xmin, r.core_ymin, r.core_xmax, r.core_ymax)
                for r in tile_df.itertuples()
            ],
            crs=crs,
        )
        gdf.to_file(footprint_gpkg, layer=footprint_layer, driver="GPKG")

    print(f"Total output tiles: {len(tile_df)}")
    print(f"Total points processed: {total_points_seen}")
    return tile_df

# Call function to tile the LiDAR data
tile_index = tile_las_by_bounds_buffered(
    input_path=LIDAR_INPUT_PATH,
    output_dir=TILE_DIR,
    tile_width=TILE_WIDTH,
    tile_height=TILE_HEIGHT,
    tile_buffer=TILE_BUFFER,
    chunk_size=1_000_000,
    min_points_per_tile=2,
    metadata_csv=TILE_METADATA_CSV,
    footprint_gpkg=TILE_FOOTPRINT_GPKG,
    footprint_layer=TILE_FOOTPRINT_LAYER,
    crs=LIDAR_CRS_STR,
)

x range: 759340.7433574634 to 759840.7423574633
y range: 5148495.495858612 to 5148995.494858611
tile grid: 2 x 2
tile buffer: 25
Processing chunk 1 with 1000000 points...
Processing chunk 2 with 1000000 points...
Processing chunk 3 with 1000000 points...
Processing chunk 4 with 1000000 points...
Processing chunk 5 with 256603 points...
Total output tiles: 4
Total points processed: 4256603


## Per‑tile LiDAR processing functions (Preprocess → CSF → Rasterize)

### Pre‑processing: Local outlier filtering and format preparation

This step applies a light preprocessing pass to each LiDAR tile to improve the stability of ground classification and raster interpolation. Rather than using only a global elevation filter, the current workflow uses a **local, spatially aware outlier filter** that compares each point’s elevation to the median elevation of its nearby neighbors and removes points that deviate beyond a configurable threshold. Optional global elevation limits can also be applied as a broad safeguard, but the main goal is to remove obvious noise while preserving valid terrain, buildings, and vegetation structure.

### Ground / non‑ground classification using Cloth Simulation Filtering (CSF)

This step classifies LiDAR points into ground and non‑ground classes using the Cloth Simulation Filtering (CSF) algorithm, an established method for terrain extraction from airborne LiDAR data. CSF works by simulating a cloth surface draped over an inverted point cloud; points that lie close to the simulated cloth are classified as ground, while points above that surface are classified as non‑ground objects such as vegetation and buildings.

The implementation uses **CSF‑3DFin** (pip install CSF-3DFin) and follows the methodology described by [Zhang et al. (2016)](https://www.mdpi.com/2072-4292/8/6/501). The workflow supports named CSF presets (for example, default, urban, forest, and steep terrain), which control two key parameters:

* cloth_resolution, which controls the smoothness of the simulated cloth surface, and
* class_threshold, which controls the vertical tolerance for assigning points to the ground surface.

Ground points are written to LAS class 2 and non‑ground points to class 1, following common LAS classification conventions. The notebook also records simple tile‑level classification summaries, including ground and non‑ground point counts and percentages, to support quick QA of CSF performance. 

Reference:
Zhang, W., Qi, J., Wan, P., Wang, H., Xie, D., Wang, X., & Yan, G. (2016). An Easy‑to‑Use Airborne LiDAR Data Filtering Algorithm Based on Cloth Simulation. Remote Sensing, 8(6), 501. https://www.mdpi.com/2072-4292/8/6/501

### Rasterization to DEM and DSM using Whitebox Workflows

This step converts each classified LiDAR tile into raster terrain and surface products using Whitebox Workflows’ LiDAR‑native interpolation routines. Rasterization is performed using **TIN‑based gridding** (Delaunay triangulation), a widely used LiDAR interpolation method for generating elevation surfaces from irregular point clouds.

A **Digital Elevation Model (DEM)** is generated using user-defined ground-supporting returns and class exclusions, while a **Digital Surface Model (DSM)** is generated using a broader return set with separate class filters to better represent the upper surface of buildings and vegetation. These DEM and DSM tiles are then passed forward to the mosaicking, alignment, and DEM void-filling steps used to create the final raster products.



In [6]:
# ----------------------------------------------------------
# Function: preprocess a single tile (light outlier removal)
# ----------------------------------------------------------

def preprocess_tile_lidar(tile_path, out_dir):
    las = laspy.read(tile_path)

    x = np.asarray(las.x)
    y = np.asarray(las.y)
    z = np.asarray(las.z)

    keep_mask = np.ones(len(z), dtype=bool)

    # Optional broad global guardrails
    if PREPROCESS_GLOBAL_ZMIN is not None:
        keep_mask &= (z >= PREPROCESS_GLOBAL_ZMIN)
    if PREPROCESS_GLOBAL_ZMAX is not None:
        keep_mask &= (z <= PREPROCESS_GLOBAL_ZMAX)

    if PREPROCESS_USE_LOCAL_FILTER and len(z) > PREPROCESS_K:
        xy = np.column_stack((x, y))
        tree = cKDTree(xy)

        # k includes the point itself
        _, idx = tree.query(xy, k=PREPROCESS_K + 1)
        neighbor_idx = idx[:, 1:]  # exclude self

        neighbor_z = z[neighbor_idx]
        local_median = np.median(neighbor_z, axis=1)
        local_mad = np.median(np.abs(neighbor_z - local_median[:, None]), axis=1)

        dz = np.abs(z - local_median)

        # robust threshold: absolute threshold OR MAD-scaled threshold
        robust_thresh = np.maximum(
            PREPROCESS_LOCAL_Z_THRESH,
            PREPROCESS_MAD_MULTIPLIER * 1.4826 * local_mad
        )

        keep_mask &= (dz <= robust_thresh)

    las_clean = laspy.LasData(las.header)
    las_clean.points = las.points[keep_mask]

    base = os.path.basename(tile_path)
    if base.lower().endswith(".laz"):
        out_name = base[:-4] + "_pre.laz"
    elif base.lower().endswith(".las"):
        out_name = base[:-4] + "_pre.las"
    else:
        out_name = base + "_pre.las"

    out_path = os.path.join(out_dir, out_name)
    las_clean.write(out_path)

    print(f"Preprocess kept {keep_mask.sum():,} / {len(keep_mask):,} points")
    return out_path

# ----------------------------------------------------------
# Function: CSF ground classification on one tile
# ----------------------------------------------------------

def classify_tile_csf(tile_path):
    las = laspy.read(tile_path)

    xyz = np.column_stack([
        np.asarray(las.x),
        np.asarray(las.y),
        np.asarray(las.z),
    ])

    csf = CSF.CSF()
    csf.params.cloth_resolution = CSF_CLOTH_RESOLUTION
    csf.params.class_threshold = CSF_CLASS_THRESHOLD

    csf.set_point_cloud(xyz)
    ground_idx, nonground_idx = csf.classify_ground()

    classification = np.asarray(las.classification).copy()
    classification[:] = 1
    classification[ground_idx] = 2
    classification[nonground_idx] = 1
    las.classification = classification

    out_path = tile_path.replace("_pre.laz", "_csf.laz").replace("_pre.las", "_csf.las")
    las.write(out_path)

    total_points = len(classification)
    ground_count = int(np.sum(classification == 2))
    nonground_count = int(np.sum(classification == 1))
    unclassified_count = int(total_points - ground_count - nonground_count)

    summary = {
        "tile_name": os.path.basename(out_path),
        "total_points": total_points,
        "ground_count": ground_count,
        "nonground_count": nonground_count,
        "unclassified_count": unclassified_count,
        "ground_pct": 100.0 * ground_count / total_points if total_points else 0.0,
        "nonground_pct": 100.0 * nonground_count / total_points if total_points else 0.0,
        "cloth_resolution": CSF_CLOTH_RESOLUTION,
        "class_threshold": CSF_CLASS_THRESHOLD,
        "csf_preset": CSF_ACTIVE_PRESET,
    }

    return out_path, summary

# ----------------------------------------------------------
# Function: rasterize DEM / DSM for one tile
# ----------------------------------------------------------

def rasterize_tile(tile_las_path, raster_dir):
    wbe = wbw.WbEnvironment()
    wbe.verbose = False
    wbe.working_directory = raster_dir

    rd_lidar = wbe.read_lidar(tile_las_path)

    tile_name = os.path.splitext(os.path.basename(tile_las_path))[0]
    dem_path = os.path.join(raster_dir, f"{tile_name}_dem.tif")
    dsm_path = os.path.join(raster_dir, f"{tile_name}_dsm.tif")

    res = float(RASTER_RESOLUTION)

    # DEM: use classified ground-supporting points only
    dem_raster = wbe.lidar.interpolation_gridding.lidar_tin_gridding(
        input=rd_lidar,
        resolution=res,
        interpolation_parameter="elevation",
        returns_included=DEM_RETURNS_INCLUDED,
        excluded_classes=DEM_EXCLUDED_CLASSES,
    )

    # DSM: use broader return set, but exclude obvious noise classes
    dsm_raster = wbe.lidar.interpolation_gridding.lidar_tin_gridding(
        input=rd_lidar,
        resolution=res,
        interpolation_parameter="elevation",
        returns_included=DSM_RETURNS_INCLUDED,
        excluded_classes=DSM_EXCLUDED_CLASSES,
    )

    wbe.write_raster(
        dem_raster,
        dem_path
    )
    wbe.write_raster(
        dsm_raster,
        dsm_path
    )

    return dem_path, dsm_path


# ----------------------------------------------------------
# Function: Clean up memory between tiles
# ----------------------------------------------------------

def cleanup():
    gc.collect()

## Tile processing loop (Preprocess → CSF → Rasterize)

Runs the per-tile processing workflow by applying preprocessing, CSF ground classification, and DEM/DSM rasterization to each buffered LiDAR tile in sequence. This step also organizes intermediate outputs, suppresses noisy library output where needed, and records tile-level classification summaries for quick QA and troubleshooting.

In [7]:
# ----------------------------------------------------------
# Tile processing loop
# Runs preprocess → CSF → rasterize for each tile
# ----------------------------------------------------------

@contextlib.contextmanager
def suppress_fd_output():
    """
    Suppress *all* output to stdout/stderr (C-level) within the context.
    This will hide noisy native library prints (e.g., CSF).
    """
    # Save original file descriptors
    old_stdout_fd = os.dup(1)
    old_stderr_fd = os.dup(2)

    try:
        # Open devnull
        with open(os.devnull, "w") as devnull:
            # Redirect stdout & stderr to devnull at OS level
            os.dup2(devnull.fileno(), 1)
            os.dup2(devnull.fileno(), 2)

            yield
    finally:
        # Restore original stdout/stderr fds
        os.dup2(old_stdout_fd, 1)
        os.dup2(old_stderr_fd, 2)
        os.close(old_stdout_fd)
        os.close(old_stderr_fd)

# ----------------------------------------------------------
# Collect tile files
# ----------------------------------------------------------

tile_files = sorted(
    os.path.join(TILE_DIR, f)
    for f in os.listdir(TILE_DIR)
    if f.lower().endswith((".las", ".laz"))
)

print(f"Found {len(tile_files)} tiles to process.")

if len(tile_files) == 0:
    raise RuntimeError("No tiles found — cannot proceed.")

# ----------------------------------------------------------
# Output subdirectories (kept simple and explicit)
# ----------------------------------------------------------

TILE_PRE_DIR = os.path.join(TILE_DIR, "preprocessed")
TILE_CSF_DIR = os.path.join(TILE_DIR, "csf")
TILE_RASTER_DIR = os.path.join(RASTER_DIR, "tiles")

os.makedirs(TILE_PRE_DIR, exist_ok=True)
os.makedirs(TILE_CSF_DIR, exist_ok=True)
os.makedirs(TILE_RASTER_DIR, exist_ok=True)

# ----------------------------------------------------------
# Process tiles one by one
# ----------------------------------------------------------

processed = 0
failed = []
tile_class_summaries = []

for tile_path in tqdm(tile_files, desc="Processing tiles"):
    tile_name = os.path.basename(tile_path)

    try:
        print(f"\n▶ Processing tile: {tile_name}")

        # 1. Preprocess (light outlier removal)
        pre_tile = preprocess_tile_lidar(
            tile_path,
            out_dir=TILE_PRE_DIR
        )

        with suppress_fd_output():
            # 2. CSF ground classification
            csf_tile, class_summary = classify_tile_csf(pre_tile)
            tile_class_summaries.append(class_summary)

            # Move CSF output to CSF directory
            csf_out_path = os.path.join(
                TILE_CSF_DIR,
                os.path.basename(csf_tile)
            )
            os.replace(csf_tile, csf_out_path)

            # 3. Rasterize DEM / DSM
            dem_path, dsm_path = rasterize_tile(
                csf_out_path,
                raster_dir=TILE_RASTER_DIR
            )

        processed += 1

    except Exception as e:
        print(f"❌ Failed tile: {tile_name}")
        print(e)
        failed.append(tile_name)

    finally:
        cleanup()   # force garbage collection between tiles

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

if tile_class_summaries:
    csf_summary_df = pd.DataFrame(tile_class_summaries)

    print("\nCSF tile-level summary:")
    display(csf_summary_df)

    csf_summary_csv = os.path.join(RASTER_DIR, "csf_tile_class_summary.csv")
    csf_summary_df.to_csv(csf_summary_csv, index=False)

    print(f"✅ CSF tile summary written: {csf_summary_csv}")
print("\n✅ Tile processing complete.")
print(f"Tiles processed successfully: {processed}")
print(f"Tiles failed: {len(failed)}")

if failed:
    print("Failed tiles:")
    for f in failed:
        print("  ", f)

Found 4 tiles to process.


Processing tiles:   0%|          | 0/4 [00:00<?, ?it/s]


▶ Processing tile: tile_0_0_buf.las
Preprocess kept 1,304,530 / 1,335,920 points


Processing tiles:  25%|██▌       | 1/4 [00:07<00:23,  7.99s/it]


▶ Processing tile: tile_0_1_buf.las
Preprocess kept 1,479,612 / 1,498,862 points


Processing tiles:  50%|█████     | 2/4 [00:15<00:15,  7.79s/it]

... (3 outputs left out)

Processing tiles: 100%|██████████| 4/4 [00:28<00:00,  7.19s/it]


CSF tile-level summary:


,tile_name,total_points,ground_count,nonground_count,unclassified_count,ground_pct,nonground_pct,cloth_resolution,class_threshold,csf_preset
0,tile_0_0_buf_csf.las,1304530,75179,1229351,0,5.762918,94.237082,1.0,0.05,default
1,tile_0_1_buf_csf.las,1479612,434029,1045583,0,29.333974,70.666026,1.0,0.05,default
2,tile_1_0_buf_csf.las,1150630,405452,745178,0,35.237392,64.762608,1.0,0.05,default
3,tile_1_1_buf_csf.las,1172448,469198,703250,0,40.018662,59.981338,1.0,0.05,default


✅ CSF tile summary written: /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/processed/rasters/csf_tile_class_summary.csv

✅ Tile processing complete.
Tiles processed successfully: 4
Tiles failed: 0


## Mosaic raster tiles and derive final surfaces

Mosaics the per-tile DEM and DSM rasters back into seamless final surfaces and checks that the DSM is aligned to the DEM grid before downstream calculations. The workflow writes a **raw DEM, final DSM, DEM fill mask, filled DEM, and nDSM**, making it easier to compare observed versus interpolated terrain surfaces and to track where void filling occurred.

Residual DEM gaps, such as those caused by canopy occlusion or sparse ground returns, are identified from the raw DEM and filled using a simple nearest-neighbor raster-domain interpolation as a first-pass void-filling method. The nDSM is then calculated from either the raw or filled DEM, depending on the user-setting, which allows both versions to be reviewed and compared before selecting the preferred output for analysis.

In [8]:
print("Starting raster mosaicking (rioxarray)...")

tile_raster_dir = os.path.join(RASTER_DIR, "tiles")

dem_tile_paths = sorted(
    os.path.join(tile_raster_dir, f)
    for f in os.listdir(tile_raster_dir)
    if f.endswith("_dem.tif")
)

dsm_tile_paths = sorted(
    os.path.join(tile_raster_dir, f)
    for f in os.listdir(tile_raster_dir)
    if f.endswith("_dsm.tif")
)

print(f"DEM tiles found: {len(dem_tile_paths)}")
print(f"DSM tiles found: {len(dsm_tile_paths)}")

if not dem_tile_paths or not dsm_tile_paths:
    raise RuntimeError("Missing DEM or DSM tiles")

def mosaic_rasters(paths):
    rasters = [
        rxr.open_rasterio(p, masked=True).rio.write_crs(LIDAR_CRS_STR)
        for p in paths
    ]    
    mosaic = merge_arrays(rasters).squeeze(drop=True)
    return mosaic

def rasters_match_grid(a, b):
    a_crs = str(a.rio.crs)
    b_crs = str(b.rio.crs)

    a_res = tuple(round(abs(v), 12) for v in a.rio.resolution())
    b_res = tuple(round(abs(v), 12) for v in b.rio.resolution())

    a_shape = a.shape
    b_shape = b.shape

    a_transform = tuple(round(v, 12) for v in a.rio.transform())
    b_transform = tuple(round(v, 12) for v in b.rio.transform())

    return (
        a_crs == b_crs and
        a_res == b_res and
        a_shape == b_shape and
        a_transform == b_transform
    )

def align_to_match_grid(source_da, target_da):
    return source_da.rio.reproject_match(
        target_da,
        resampling=Resampling.nearest
    )

def prepare_for_raster_write(da):
    da = da.copy()
    da.attrs.pop("_FillValue", None)
    return da

# ----------------------------------------------------------
# Mosaic DEM and DSM
# ----------------------------------------------------------

dem_raw_da = mosaic_rasters(dem_tile_paths)
dem_raw_da = dem_raw_da.rio.write_crs(LIDAR_CRS_STR, inplace=False)

dsm_da = mosaic_rasters(dsm_tile_paths)
dsm_da = dsm_da.rio.write_crs(LIDAR_CRS_STR, inplace=False)

if rasters_match_grid(dem_raw_da, dsm_da):
    print("✅ DEM and DSM grids already match")
else:
    print("⚠️ DEM and DSM grids differ; aligning DSM to DEM grid")
    dsm_da = align_to_match_grid(dsm_da, dem_raw_da)

    if rasters_match_grid(dem_raw_da, dsm_da):
        print("✅ DSM successfully aligned to DEM grid")
    else:
        raise ValueError("DSM could not be aligned to DEM grid")

# ----------------------------------------------------------
# Check raster alignment
# ----------------------------------------------------------

def check_raster_alignment(da_dict, tol=1e-9):
    """
    Quick alignment check for mosaicked rasters.
    Confirms CRS, resolution, shape, and affine transform match.
    """
    names = list(da_dict.keys())
    ref_name = names[0]
    ref = da_dict[ref_name]

    ref_crs = str(ref.rio.crs)
    ref_res = tuple(round(abs(v), 12) for v in ref.rio.resolution())
    ref_shape = ref.shape
    ref_transform = tuple(round(v, 12) for v in ref.rio.transform())

    for name in names[1:]:
        da = da_dict[name]
        crs = str(da.rio.crs)
        res = tuple(round(abs(v), 12) for v in da.rio.resolution())
        shape = da.shape
        transform = tuple(round(v, 12) for v in da.rio.transform())

        if crs != ref_crs:
            raise ValueError(f"CRS mismatch: {name} has {crs}, expected {ref_crs}")
        if res != ref_res:
            raise ValueError(f"Resolution mismatch: {name} has {res}, expected {ref_res}")
        if shape != ref_shape:
            raise ValueError(f"Shape mismatch: {name} has {shape}, expected {ref_shape}")
        if transform != ref_transform:
            raise ValueError(
                f"Transform mismatch: {name} is not on the same grid as {ref_name}"
            )

    print("✅ Raster alignment check passed: CRS, resolution, shape, and transform match")

# ----------------------------------------------------------
# Save raw DEM
# ----------------------------------------------------------

dem_raw_path = os.path.join(RASTER_DIR, "dem_raw.tif")
dsm_final_path = os.path.join(RASTER_DIR, "dsm_final.tif")

if WRITE_RAW_DEM:
    prepare_for_raster_write(dem_raw_da).rio.to_raster(
        dem_raw_path,
        compress="LZW",
        dtype="float32",
    )
    print("✅ Raw DEM written")

# ----------------------------------------------------------
# Fill DSM voids
# ----------------------------------------------------------

dsm_filled_da = dsm_da.copy()

dsm_arr = dsm_filled_da.values
dsm_mask = np.isnan(dsm_arr)

if np.any(dsm_mask):
    nearest_idx = distance_transform_edt(
        dsm_mask,
        return_distances=False,
        return_indices=True
    )
    dsm_arr_filled = dsm_arr[tuple(nearest_idx)]

    dsm_filled_da.values = dsm_arr_filled
    print("✅ DSM voids filled (nearest-neighbor first pass)")
else:
    print("✅ No DSM voids detected; fill step skipped")

prepare_for_raster_write(dsm_filled_da).rio.to_raster(
    dsm_final_path,
    compress="LZW",
    dtype="float32",
)
print("✅ Final DSM written")

# ----------------------------------------------------------
# Build void mask from raw DEM
# ----------------------------------------------------------

raw_nodata_mask = dem_raw_da.isnull()
fill_mask_da = raw_nodata_mask.astype("uint8")

fill_mask_path = os.path.join(RASTER_DIR, "dem_fill_mask.tif")
if WRITE_FILL_MASK:
    fill_mask_da.rio.write_crs(LIDAR_CRS_STR, inplace=True)
    prepare_for_raster_write(fill_mask_da).rio.to_raster(
        fill_mask_path,
        compress="LZW",
        dtype="uint8",
    )
    print("✅ DEM fill mask written")

# ----------------------------------------------------------
# Fill DEM voids
# ----------------------------------------------------------

dem_filled_da = dem_raw_da.copy()

arr = dem_filled_da.values
mask = np.isnan(arr)

if np.any(mask):
    # nearest-neighbor fill as first-pass gap fill
    nearest_idx = distance_transform_edt(mask, return_distances=False, return_indices=True)
    arr_filled = arr[tuple(nearest_idx)]

    dem_filled_da.values = arr_filled
    print("✅ DEM voids filled (nearest-neighbor first pass)")
else:
    print("✅ No DEM voids detected; fill step skipped")

dem_filled_path = os.path.join(RASTER_DIR, "dem_filled.tif")
if WRITE_FILLED_DEM:
    prepare_for_raster_write(dem_filled_da).rio.to_raster(
        dem_filled_path,
        compress="LZW",
        dtype="float32",
    )
    print("✅ Filled DEM written")

# ----------------------------------------------------------
# Choose DEM source for nDSM
# ----------------------------------------------------------

if NDSM_SOURCE_DEM == "filled":
    dem_for_ndsm = dem_filled_da
    print("Using filled DEM for nDSM")
elif NDSM_SOURCE_DEM == "raw":
    dem_for_ndsm = dem_raw_da
    print("Using raw DEM for nDSM")
else:
    raise ValueError("NDSM_SOURCE_DEM must be 'raw' or 'filled'")

# ----------------------------------------------------------
# Create nDSM
# ----------------------------------------------------------

ndsm_da = dsm_filled_da - dem_for_ndsm
ndsm_da = ndsm_da.where(ndsm_da >= 0, 0)

ndsm_path = os.path.join(RASTER_DIR, "ndsm_final.tif")
prepare_for_raster_write(ndsm_da).rio.to_raster(
    ndsm_path,
    compress="LZW",
    dtype="float32",
)
print("✅ Final nDSM written")

# ----------------------------------------------------------
# Basic fill stats
# ----------------------------------------------------------

fill_pixels = int(fill_mask_da.values.sum())
total_pixels = int(fill_mask_da.size)
fill_pct = 100.0 * fill_pixels / total_pixels if total_pixels else 0.0

print(f"Filled DEM pixels: {fill_pixels:,}")
print(f"Percent of DEM filled: {fill_pct:.3f}%")

# Close handles
dem_raw_da.close()
dsm_da.close()
dsm_filled_da.close()
dem_filled_da.close()
fill_mask_da.close()
ndsm_da.close()


Starting raster mosaicking (rioxarray)...
DEM tiles found: 4
DSM tiles found: 4
⚠️ DEM and DSM grids differ; aligning DSM to DEM grid
✅ DSM successfully aligned to DEM grid
✅ Raw DEM written
✅ DSM voids filled (nearest-neighbor first pass)
✅ Final DSM written
✅ DEM fill mask written
✅ DEM voids filled (nearest-neighbor first pass)
✅ Filled DEM written
Using filled DEM for nDSM
✅ Final nDSM written
Filled DEM pixels: 270
Percent of DEM filled: 0.108%


## QA checks and summaries

Generates lightweight numerical QA summaries for the final raster products without creating plots or loading unnecessary intermediate data. This step reports basic elevation statistics, NoData coverage, DEM fill extent, and simple nDSM sanity checks, and writes the results to a CSV for later review. Because it reopens only the final outputs, it is better suited to large datasets than notebook-based visualization.

In [9]:
qa_rows = []

dem_raw_path = os.path.join(RASTER_DIR, "dem_raw.tif")
dem_filled_path = os.path.join(RASTER_DIR, "dem_filled.tif")
dsm_final_path = os.path.join(RASTER_DIR, "dsm_final.tif")
ndsm_path = os.path.join(RASTER_DIR, "ndsm_final.tif")
fill_mask_path = os.path.join(RASTER_DIR, "dem_fill_mask.tif")

def raster_qa_stats(raster_path, raster_name):
    da = rxr.open_rasterio(raster_path, masked=True).squeeze(drop=True)
    arr = da.values

    valid_mask = np.isfinite(arr)
    nodata_mask = ~valid_mask

    valid = arr[valid_mask]
    total_cells = arr.size
    valid_cells = int(valid_mask.sum())
    nodata_cells = int(nodata_mask.sum())
    nodata_pct = 100.0 * nodata_cells / total_cells if total_cells else 0.0

    row = {
        "raster": raster_name,
        "rows": int(arr.shape[0]),
        "cols": int(arr.shape[1]),
        "valid_cells": valid_cells,
        "nodata_cells": nodata_cells,
        "nodata_pct": nodata_pct,
        "min": float(np.nanmin(valid)) if valid.size else np.nan,
        "max": float(np.nanmax(valid)) if valid.size else np.nan,
        "mean": float(np.nanmean(valid)) if valid.size else np.nan,
        "median": float(np.nanmedian(valid)) if valid.size else np.nan,
        "p01": float(np.nanpercentile(valid, 1)) if valid.size else np.nan,
        "p99": float(np.nanpercentile(valid, 99)) if valid.size else np.nan,
    }

    da.close()
    return row

# ----------------------------------------------------------
# Basic raster summaries
# ----------------------------------------------------------

for path, name in [
    (dem_raw_path, "dem_raw"),
    (dem_filled_path, "dem_filled"),
    (dsm_final_path, "dsm_final"),
    (ndsm_path, "ndsm_final"),
]:
    if os.path.exists(path):
        qa_rows.append(raster_qa_stats(path, name))

# ----------------------------------------------------------
# DEM fill-mask summary
# ----------------------------------------------------------

if os.path.exists(fill_mask_path):
    fill_mask_da = rxr.open_rasterio(fill_mask_path, masked=True).squeeze(drop=True)
    fill_mask = fill_mask_da.values

    filled_cells = int(np.nansum(fill_mask))
    total_cells = fill_mask.size
    filled_pct = 100.0 * filled_cells / total_cells if total_cells else 0.0

    qa_rows.append({
        "raster": "dem_fill_mask",
        "rows": int(fill_mask.shape[0]),
        "cols": int(fill_mask.shape[1]),
        "valid_cells": int(np.isfinite(fill_mask).sum()),
        "nodata_cells": int((~np.isfinite(fill_mask)).sum()),
        "nodata_pct": 100.0 * int((~np.isfinite(fill_mask)).sum()) / total_cells if total_cells else 0.0,
        "min": float(np.nanmin(fill_mask)),
        "max": float(np.nanmax(fill_mask)),
        "mean": float(np.nanmean(fill_mask)),
        "median": float(np.nanmedian(fill_mask)),
        "p01": float(np.nanpercentile(fill_mask, 1)),
        "p99": float(np.nanpercentile(fill_mask, 99)),
        "filled_cells": filled_cells,
        "filled_pct": filled_pct,
    })

    fill_mask_da.close()
else:
    filled_cells = np.nan
    filled_pct = np.nan

# ----------------------------------------------------------
# nDSM sanity checks
# ----------------------------------------------------------

ndsm_negative_summary = {}
if os.path.exists(ndsm_path):
    ndsm_da = rxr.open_rasterio(ndsm_path, masked=True).squeeze(drop=True)
    ndsm_arr = ndsm_da.values

    valid_ndsm = ndsm_arr[np.isfinite(ndsm_arr)]
    negative_mask = valid_ndsm < 0

    ndsm_negative_summary = {
        "ndsm_negative_cells": int(negative_mask.sum()),
        "ndsm_negative_pct": 100.0 * int(negative_mask.sum()) / len(valid_ndsm) if len(valid_ndsm) else 0.0,
        "ndsm_min": float(np.nanmin(valid_ndsm)) if len(valid_ndsm) else np.nan,
        "ndsm_max": float(np.nanmax(valid_ndsm)) if len(valid_ndsm) else np.nan,
    }

    ndsm_da.close()

# ----------------------------------------------------------
# Write QA summary
# ----------------------------------------------------------

qa_df = pd.DataFrame(qa_rows)

for col, value in ndsm_negative_summary.items():
    qa_df.loc[qa_df["raster"] == "ndsm_final", col] = value

qa_csv_path = os.path.join(RASTER_DIR, "qa_raster_summary.csv")
qa_df.to_csv(qa_csv_path, index=False)

print("\nQA raster summary")
display(qa_df)

print(f"\n✅ QA summary written: {qa_csv_path}")

if not np.isnan(filled_cells):
    print(f"Filled DEM cells: {filled_cells:,}")
    print(f"Percent of raster filled: {filled_pct:.3f}%")

if ndsm_negative_summary:
    print(f"nDSM negative cells: {ndsm_negative_summary['ndsm_negative_cells']:,}")
    print(f"nDSM negative percent: {ndsm_negative_summary['ndsm_negative_pct']:.3f}%")


QA raster summary


,raster,rows,cols,valid_cells,nodata_cells,nodata_pct,min,max,mean,median,p01,p99,filled_cells,filled_pct,ndsm_negative_cells,ndsm_negative_pct,ndsm_min,ndsm_max
0,dem_raw,500,500,249730,270,0.108,761.995972,911.406555,820.822510,810.992493,764.091736,900.344849,NaN,NaN,NaN,NaN,NaN,NaN
1,dem_filled,500,500,250000,0,0.000,761.995972,911.406555,820.894470,811.029358,764.091980,900.509399,NaN,NaN,NaN,NaN,NaN,NaN
2,dsm_final,500,500,250000,0,0.000,761.945557,929.426758,825.321533,815.741333,765.329102,911.958801,NaN,NaN,NaN,NaN,NaN,NaN
3,ndsm_final,500,500,250000,0,0.000,0.000000,36.980663,4.431107,0.803309,0.000000,25.686155,NaN,NaN,0.0,0.0,0.0,36.980663
4,dem_fill_mask,500,500,250000,0,0.000,0.000000,1.000000,0.001080,0.000000,0.000000,0.000000,270.0,0.108,NaN,NaN,NaN,NaN



✅ QA summary written: /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/processed/rasters/qa_raster_summary.csv
Filled DEM cells: 270
Percent of raster filled: 0.108%
nDSM negative cells: 0
nDSM negative percent: 0.000%


## Cleanup intermediate files
Removes temporary rasters and intermediate tile-processing files created during preprocessing, classification, and raster generation to keep the workspace organized and avoid file-lock or sync issues in cloud-backed folders such as OneDrive. This section can be commented out when intermediate files need to be preserved for review, debugging, or testing of individual workflow steps.

In [10]:
# ----------------------------------------------------------
# Function: remove directory contents safely (OneDrive‑safe)
# ----------------------------------------------------------

def clear_directory_contents(path):
    if not os.path.exists(path):
        return

    for name in os.listdir(path):
        full_path = os.path.join(path, name)
        try:
            if os.path.isdir(full_path):
                shutil.rmtree(full_path)
            else:
                os.remove(full_path)
        except PermissionError as e:
            print(f"⚠️ Could not delete {full_path}: {e}")

# ----------------------------------------------------------
# 1. Remove all *_tmp.tif files under RASTER_DIR
# ----------------------------------------------------------

for root, _, files in os.walk(RASTER_DIR):
    for f in files:
        if f.endswith("_tmp.tif"):
            tmp_path = os.path.join(root, f)
            try:
                os.remove(tmp_path)
            except PermissionError as e:
                print(f"⚠️ Could not delete {tmp_path}: {e}")

print("✅ Removed all *_tmp.tif files under RASTER_DIR")

# ----------------------------------------------------------
# 2. Clear raster/tiles contents
# ----------------------------------------------------------

print(f"Clearing contents of {tile_raster_dir}")
clear_directory_contents(tile_raster_dir)

# ----------------------------------------------------------
# 3. Clear TILE_CSF_DIR contents
# ----------------------------------------------------------

print(f"Clearing contents of {TILE_CSF_DIR}")
clear_directory_contents(TILE_CSF_DIR)

# ----------------------------------------------------------
# 4. Clear TILE_PRE_DIR contents
# ----------------------------------------------------------

print(f"Clearing contents of {TILE_PRE_DIR}")
clear_directory_contents(TILE_PRE_DIR)

print("✅ Cleanup complete")

✅ Removed all *_tmp.tif files under RASTER_DIR
Clearing contents of /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/processed/rasters/tiles
Clearing contents of /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/raw/lidar_tiles/csf
Clearing contents of /Users/pamelagreen/Desktop/Career/Python/lidar_dem_dsm_workflow/data/raw/lidar_tiles/preprocessed
✅ Cleanup complete


## Runtime Summary
Stops the workflow timer and reports the total processing time in seconds and minutes. This provides a simple benchmark for comparing runs and can be commented out if runtime tracking is not needed.

In [11]:
end = time.time()
elapsed = end - start

minutes = int(elapsed // 60)
seconds = elapsed % 60

print(f"Total runtime: {end - start:.2f} seconds")
print(f"Total runtime: {minutes} minutes {seconds:.2f} seconds")

Total runtime: 29.73 seconds
Total runtime: 0 minutes 29.73 seconds
